In [ ]:
import torch

B = 16
T = 9
S = 10
F = 768

sequence = torch.randn(B, T, S)
sequence = sequence.moveaxis(1, 2).unsqueeze(-1) # B, S, T, 1

def positional_encoding(
        d_model: int,
        t: torch.Tensor,
    ) -> torch.Tensor:
    """
    Args:
    - d_model: int
    - t: torch.Tensor (shape [batch_size, sequence_length])

    Returns: torch.Tensor (shape [batch_size, sequence_length, d_model])
    """
    inv_freq = 1.0 / (
        10000
        ** (torch.arange(0, d_model, 2, device=t.device) / d_model)
    )
    # Ensure `t` has shape [batch_size, sequence_length, 1]
    t = t.unsqueeze(-1)  # Shape [batch_size, sequence_length, 1]
    pos_enc_a = torch.sin(t * inv_freq)  # Shape [batch_size, sequence_length, d_model // 2]
    pos_enc_b = torch.cos(t * inv_freq)  # Shape [batch_size, sequence_length, d_model // 2]
    pos_enc = torch.cat([pos_enc_a, pos_enc_b], dim=-1)  # Shape [batch_size, sequence_length, d_model]
    return pos_enc

embedders = torch.nn.ModuleList(
    torch.nn.Sequential(
        torch.nn.Linear(1, F),
        torch.nn.BatchNorm1d(F),
        torch.nn.ReLU(),
        torch.nn.Linear(F, F),
        torch.nn.BatchNorm1d(F),
        torch.nn.ReLU(),
    ) # (B, S, T, 1) x (1, F) = (B, S, T, F)
 for _ in range(S))

# Featurize each source independently
features = []
for si in range(S):
    s = sequence[:, si, :, :] # (B, T, 1)

    featurizer_si = embedders[si]
    f = featurizer_si(s) # (B, T, 1) x (1, F) = (B, T, F)

    ts = torch.range(0, T-1, device=s.device)
    pos_encodings = positional_encoding(F, ts) # (T, F)
    pos_encodings = pos_encodings.unsqueeze(0) # (1, T, F) || (B, T, F)
    f = f + pos_encodings # (B, T, F)
    features.append(f)

features = torch.stack(features, dim=1) # (B, S, T, F)

print("features", features.shape)

combined = features.reshape(B, S*T, F) # (B, S*T, F)
combined = combiner(combined)
result = combined[:, -T:, :]
result.shape

torch.Size([16, 9, 10])